In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

drive.mount("/content/drive")

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH, "data", "processed"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH, "reports", "figures"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH, "reports", "tables"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH, "outputs"
)

os.makedirs(FIGURE_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

clean_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase3_clean.parquet"
    )
)

clean_df["Datetime"] = pd.to_datetime(
    clean_df["Datetime"]
)

print("Dataset restored.")
print("Shape:", clean_df.shape)

In [ ]:
print(clean_df.columns.tolist())
print()
clean_df.info()

In [ ]:
station_summary = (
    clean_df
    .groupby(["StationId", "StationName"])
    .agg(
        Records=("Datetime", "size"),
        Start=("Datetime", "min"),
        End=("Datetime", "max"),
        MeanAQI=("AQI", "mean"),
        MaxAQI=("AQI", "max")
    )
    .sort_values("Records", ascending=False)
)

display(station_summary)

In [ ]:
plt.figure(figsize=(14, 8))

station_counts = (
    clean_df["StationName"]
    .value_counts()
    .sort_values()
)

station_counts.plot(kind="barh")

plt.title("Usable Records by Delhi Monitoring Station")
plt.xlabel("Number of Records")
plt.ylabel("Station")
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "station_record_counts.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    clean_df["AQI"],
    bins=60
)

plt.title("Distribution of Delhi AQI")
plt.xlabel("AQI")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "aqi_distribution.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

plt.hist(
    clean_df["AQI_t_plus_1"],
    bins=60
)

plt.title("Distribution of Next-Hour AQI")
plt.xlabel("AQI at t+1")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "future_aqi_distribution.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
hourly_aqi = (
    clean_df
    .groupby("Hour")["AQI"]
    .mean()
)

plt.figure(figsize=(12, 6))

plt.plot(
    hourly_aqi.index,
    hourly_aqi.values,
    marker="o"
)

plt.title("Average AQI by Hour of Day")
plt.xlabel("Hour")
plt.ylabel("Mean AQI")
plt.xticks(range(24))
plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "aqi_by_hour.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
weekday_aqi = (
    clean_df
    .groupby("DayOfWeek")["AQI"]
    .mean()
)

day_labels = [
    "Mon", "Tue", "Wed",
    "Thu", "Fri", "Sat", "Sun"
]

plt.figure(figsize=(10, 6))

plt.bar(
    day_labels,
    weekday_aqi.values
)

plt.title("Average AQI by Day of Week")
plt.xlabel("Day")
plt.ylabel("Mean AQI")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "aqi_by_day_of_week.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
monthly_aqi = (
    clean_df
    .groupby("Month")["AQI"]
    .agg(["mean", "median", "max"])
)

display(monthly_aqi)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    monthly_aqi.index,
    monthly_aqi["mean"],
    marker="o",
    label="Mean AQI"
)

plt.plot(
    monthly_aqi.index,
    monthly_aqi["median"],
    marker="s",
    label="Median AQI"
)

plt.title("Delhi AQI by Month")
plt.xlabel("Month")
plt.ylabel("AQI")
plt.xticks(range(1, 13))
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "aqi_by_month.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
season_aqi = (
    clean_df
    .groupby("Season")["AQI"]
    .agg(
        MeanAQI="mean",
        MedianAQI="median",
        MaxAQI="max",
        Records="count"
    )
    .sort_values("MeanAQI", ascending=False)
)

display(season_aqi)

In [ ]:
station_aqi = (
    clean_df
    .groupby(["StationId", "StationName"])["AQI"]
    .agg(
        MeanAQI="mean",
        MedianAQI="median",
        MaxAQI="max"
    )
    .sort_values("MeanAQI", ascending=False)
)

display(station_aqi)

In [ ]:
plt.figure(figsize=(14, 9))

station_aqi["MeanAQI"].sort_values().plot(
    kind="barh"
)

plt.title("Average AQI by Delhi Monitoring Station")
plt.xlabel("Mean AQI")
plt.ylabel("Station")
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "mean_aqi_by_station.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
event_summary = clean_df.groupby(
    ["IsStubbleSeason", "IsWeekend"]
)["AQI"].agg(
    MeanAQI="mean",
    MedianAQI="median",
    MaxAQI="max",
    Records="count"
)

display(event_summary)

In [ ]:
extreme_summary = (
    clean_df[
        clean_df["AQI_extreme_flag"] == 1
    ]["AQI"]
    .describe()
)

display(extreme_summary)

In [ ]:
correlation_columns = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene"
]

correlation_matrix = (
    clean_df[correlation_columns]
    .corr()
)

display(
    correlation_matrix["AQI"]
    .sort_values(ascending=False)
    .to_frame("CorrelationWithAQI")
)

In [ ]:
plt.figure(figsize=(12, 10))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)

plt.title("AQI and Pollutant Correlation Matrix")
plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "correlation_matrix.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
persistence_correlation = clean_df[
    ["AQI", "AQI_t_plus_1"]
].corr().iloc[0, 1]

print(
    "Correlation between AQI(t) and AQI(t+1):",
    round(persistence_correlation, 4)
)

In [ ]:
sample = clean_df[
    ["AQI", "AQI_t_plus_1"]
].dropna()

if len(sample) > 100000:
    sample = sample.sample(
        100000,
        random_state=42
    )

plt.figure(figsize=(8, 8))

plt.scatter(
    sample["AQI"],
    sample["AQI_t_plus_1"],
    alpha=0.2,
    s=10
)

min_value = min(
    sample["AQI"].min(),
    sample["AQI_t_plus_1"].min()
)

max_value = max(
    sample["AQI"].max(),
    sample["AQI_t_plus_1"].max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.title("Current AQI vs Next-Hour AQI")
plt.xlabel("AQI at t")
plt.ylabel("AQI at t+1")

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "current_vs_next_hour_aqi.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
monthly_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "monthly_aqi_summary.csv"
    )
)

season_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "season_aqi_summary.csv"
    )
)

station_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "station_aqi_summary.csv"
    )
)

event_summary.to_csv(
    os.path.join(
        TABLE_PATH,
        "event_aqi_summary.csv"
    )
)

print("EDA tables saved successfully.")

In [ ]:
eda_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase4_eda.parquet"
)

clean_df.to_parquet(
    eda_checkpoint,
    index=False
)

print("EDA checkpoint saved:")
print(eda_checkpoint)

In [ ]:
print("=" * 60)
print("PHASE 4 — EDA SUMMARY")
print("=" * 60)

print("Records:", len(clean_df))
print("Stations:", clean_df["StationId"].nunique())
print("AQI mean:", round(clean_df["AQI"].mean(), 2))
print("AQI median:", round(clean_df["AQI"].median(), 2))
print("AQI maximum:", round(clean_df["AQI"].max(), 2))
print(
    "AQI(t) ↔ AQI(t+1) correlation:",
    round(persistence_correlation, 4)
)

print("\nMean AQI by season:")
display(season_aqi)

print("=" * 60)

In [ ]:
import os
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(PROCESSED_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save Phase 4 dataset checkpoint
# --------------------------------------------------

eda_checkpoint = os.path.join(
    PROCESSED_PATH,
    "delhi_phase4_eda.parquet"
)

clean_df.to_parquet(
    eda_checkpoint,
    index=False
)

# --------------------------------------------------
# 2. Save key EDA findings
# --------------------------------------------------

eda_summary = {
    "completed_phase": "Phase 4 - Exploratory Data Analysis",
    "next_phase": "Phase 5 - Feature Engineering",

    "records": int(len(clean_df)),
    "stations": int(clean_df["StationId"].nunique()),

    "aqi_mean": float(clean_df["AQI"].mean()),
    "aqi_median": float(clean_df["AQI"].median()),
    "aqi_max": float(clean_df["AQI"].max()),

    "aqi_t_to_t1_correlation": float(
        persistence_correlation
    ),

    "season_mean_aqi": {
        str(index): float(row["MeanAQI"])
        for index, row in season_aqi.iterrows()
    },

    "extreme_aqi_records": int(
        clean_df["AQI_extreme_flag"].sum()
    )
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase4_eda_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(
        eda_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 3. Save important EDA tables
# --------------------------------------------------

season_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase4_season_aqi.csv"
    )
)

station_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase4_station_aqi.csv"
    )
)

monthly_aqi.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase4_monthly_aqi.csv"
    )
)

event_summary.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase4_event_aqi.csv"
    )
)

# --------------------------------------------------
# 4. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 4 - Exploratory Data Analysis",
    "next_phase": "Phase 5 - Feature Engineering",
    "records": int(len(clean_df)),
    "stations": int(clean_df["StationId"].nunique()),
    "target": "AQI_t_plus_1"
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(status_path, "w") as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 5. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    eda_checkpoint,
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 4 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 4")
print("Next: Phase 5 - Feature Engineering")